In [11]:
!pip install -q segmentation-models timm

In [12]:
import os
# Must define backend before importing segmentation_models
os.environ["SM_FRAMEWORK"] = "tf.keras"

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import segmentation_models as sm
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from tqdm import tqdm

BASE_PATH = '/kaggle/input/terra-seg-rugged-terrain-segmentation/offroad-seg-kaggle'
TRAIN_IMG_DIR = os.path.join(BASE_PATH, 'train_images')
TRAIN_MASK_DIR = os.path.join(BASE_PATH, 'train_masks')
TEST_IMG_DIR = os.path.join(BASE_PATH, 'test_images_padded')

IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 16
EPOCHS = 15
BACKBONE = 'resnet34'

# Preprocessing function required for ResNet
preprocess_input = sm.get_preprocessing(BACKBONE)

print(f"✅ Configuration Set. Training on: {TRAIN_IMG_DIR}")

class TerraSegGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_ids, img_dir, mask_dir=None, batch_size=16, img_size=(256, 256), is_train=True):
        self.image_ids = image_ids
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.batch_size = batch_size
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self):
        return int(np.ceil(len(self.image_ids) / self.batch_size))

    def __getitem__(self, index):
        batch_ids = self.image_ids[index * self.batch_size : (index + 1) * self.batch_size]
        
        images = []
        masks = []
        
        for img_id in batch_ids:
            # 1. LOAD IMAGE
            img_path = os.path.join(self.img_dir, img_id)
            img = cv2.imread(img_path)
            img = cv2.resize(img, self.img_size)
            
            # 2. PREPROCESS (Standard ResNet Normalization)
            img = preprocess_input(img)
            images.append(img)
            
            if self.is_train:
                # 3. LOAD MASK
                mask_path = os.path.join(self.mask_dir, img_id) 
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                
                # Resize with Nearest Neighbor (Keep values sharp)
                mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
                
                # --- UNIVERSAL THRESHOLD FIX ---
                mask = (mask > 0).astype(np.float32)
                
                mask = np.expand_dims(mask, axis=-1)
                masks.append(mask)
        
        if self.is_train:
            return np.array(images).astype(np.float32), np.array(masks).astype(np.float32)
        else:
            return np.array(images).astype(np.float32)

all_ids = os.listdir(TRAIN_IMG_DIR)
# We define train_ids/val_ids HERE, before creating the generators
train_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42)

# Now we can safely create the generators
train_gen = TerraSegGenerator(train_ids, TRAIN_IMG_DIR, TRAIN_MASK_DIR, BATCH_SIZE, (IMG_HEIGHT, IMG_WIDTH), is_train=True)
val_gen = TerraSegGenerator(val_ids, TRAIN_IMG_DIR, TRAIN_MASK_DIR, BATCH_SIZE, (IMG_HEIGHT, IMG_WIDTH), is_train=True)

model = sm.Unet(BACKBONE, encoder_weights='imagenet', classes=1, activation='sigmoid')

callbacks = [
    ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_iou_score', mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_iou_score', factor=0.5, patience=3, min_lr=1e-6, verbose=1, mode='max'),
    EarlyStopping(monitor='val_iou_score', patience=8, restore_best_weights=True, verbose=1, mode='max')
]

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss=sm.losses.bce_jaccard_loss,
    metrics=[sm.metrics.iou_score]
)

print("🚀 Starting Training...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("\n🔍 Generating Submission...")

def rle_encode(img):
    pixels = img.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

# Load the best weights we just trained
if os.path.exists('best_model.keras'):
    print("✅ Loading best_model.keras")
    model.load_weights('best_model.keras')
else:
    print("⚠️ best_model.keras not found, using current model weights")

test_ids = sorted(os.listdir(TEST_IMG_DIR))
submission_data = []

print(f"🚀 Processing {len(test_ids)} test images...")

for img_id in tqdm(test_ids):
    path = os.path.join(TEST_IMG_DIR, img_id)
    img = cv2.imread(path)
    if img is None: continue
        
    orig_h, orig_w = img.shape[:2]
    
    # Resize & Preprocess
    img_resized = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img_batch = preprocess_input(np.array([img_resized], dtype=np.float32))
    
    # Predict
    pred_batch = model.predict(img_batch, verbose=0)
    pred_mask = pred_batch[0]
    
    # Threshold & Resize
    pred_binary = (pred_mask > 0.5).astype(np.uint8)
    final_mask = cv2.resize(pred_binary, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    
    # Encode
    rle = rle_encode(final_mask)
    
    # Clean ID
    clean_id = img_id.replace('.png', '') 
    
    submission_data.append([clean_id, rle])

# Save
df_sub = pd.DataFrame(submission_data, columns=['image_id', 'encoded_pixels'])
df_sub.to_csv('submission.csv', index=False)

print(f"\n🎉 DONE! 'submission.csv' generated with {len(df_sub)} rows.")
print(f"Sample IDs: {df_sub['image_id'].head().tolist()}")

Add and fix preprocessing and augmentation alignment.

✅ Configuration Set. Training on: /kaggle/input/terra-seg-rugged-terrain-segmentation/offroad-seg-kaggle/train_images
🚀 Starting Training...
Epoch 1/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 498ms/step - iou_score: 0.4784 - loss: 1.2494
Epoch 1: val_iou_score improved from -inf to 0.64229, saving model to best_model.keras
159/159 ━━━━━━━━━━━━━━━━━━━━ 138s 643ms/step - iou_score: 0.4790 - loss: 1.2477 - val_iou_score: 0.6423 - val_loss: 0.8005 - learning_rate: 1.0000e-04
Epoch 2/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 411ms/step - iou_score: 0.7385 - loss: 0.5653
Epoch 2: val_iou_score improved from 0.64229 to 0.78736, saving model to best_model.keras
159/159 ━━━━━━━━━━━━━━━━━━━━ 82s 514ms/step - iou_score: 0.7387 - loss: 0.5650 - val_iou_score: 0.7874 - val_loss: 0.4732 - learning_rate: 1.0000e-04
Epoch 3/15
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 407ms/step - iou_score: 0.8233 - loss: 0.3818
Epoch 3: val_iou_score improved from 0.78736 to 0.85424, saving model to best_model.keras
159/159 ━━━━━━━━━━━━━━━━

100%|██████████| 1002/1002 [01:56<00:00,  8.61it/s]


🎉 DONE! 'submission.csv' generated with 1002 rows.
